# LCDM + N_eff full-plik frequentist profile — analysis

Reads the per-POI profile-likelihood `.npz` written by `scan/profile_prod_ad.py` for the
**7-POI LCDM+N_eff full-Planck-plik** headline (job `55235706`, slurm
`scan/profile_prod_plikfull_neff48.slurm`, tag **`_pfrc`**), summarises the best-fit ± 1σ
intervals with the AD-gradient convergence certificate, and cross-checks them against
(a) the **SMC posterior** for the *same* likelihood+model+code (`smc_plikfull_neff.npz`)
and (b) **Planck 2018** literature values.

**Backstory (why this run / tag):** the first headline run (`54980729`) used LCDM-centered
grids; freeing N_eff pulls N_eff→~2.83 and the N_eff–ω_cdm–H0 degeneracy drags ω_cdm/h
down ~3σ/1.4σ, so those POIs railed at the low grid edge (ω_cdm interval came back `nan`)
and the `nan` blocked the σ1 early-stop → timeout. The second run (`55017292`) enabled the
MLE pre-pass, whose single-point full-D AD compile hung for the full 10h (zero output). This
final run uses the **proven path**: grids recentered on the joint MLE
(`configs/lcdm_neff_plikfull_rc.py`), warm-start+preconditioner at that center, the hardened
σ1 early-stop, 1 POI/node × 7 nodes, 48h walltime.

Pure `numpy`+`matplotlib` (no GPU). Run with the `actdr6` kernel. Robust to POIs still
pending — each POI_SLICE rank writes its `.npz` when it finishes, so re-run cells as the
job progresses.

In [ ]:
import os, glob
import numpy as np
import matplotlib.pyplot as plt

# ---- config -------------------------------------------------------------
_cands = ['/pscratch/sd/c/carag/ABCMB-k/scan/results',   # known absolute (NERSC)
          os.path.abspath('results'), os.path.abspath('scan/results'),
          os.path.abspath(os.path.join('..', 'results'))]
RES = next((p for p in _cands if os.path.isdir(p)), _cands[0])
PREFIX = 'profile_prod_ad'
TAG    = os.environ.get('PA_NB_TAG', '_pfrc')    # final recentered-grid run (env-override for testing)
ORDER  = ['h','omega_b','omega_cdm','n_s','ln10As','tau_reion','Neff']
LABEL  = {'h':r'$h$','omega_b':r'$\omega_b$','omega_cdm':r'$\omega_{cdm}$',
          'n_s':r'$n_s$','ln10As':r'$\ln(10^{10}A_s)$','tau_reion':r'$\tau$','Neff':r'$N_{eff}$'}
print('results dir:', RES, '| TAG:', TAG)

# ---- reference values (editable) ---------------------------------------
# Planck 2018 base-LCDM, TT,TE,EE+lowE (Planck 2018 VI). For the 6 LCDM params these are the
# standard anchor; freeing N_eff SHIFTS+WIDENS them, so expect the profile to sit off these.
PLANCK_LCDM = {'h':(0.6736,0.0054), 'omega_b':(0.02237,0.00015), 'omega_cdm':(0.1200,0.0012),
               'n_s':(0.9649,0.0042), 'ln10As':(3.044,0.014), 'tau_reion':(0.0544,0.0073)}
# Planck 2018 base_nnu (N_eff free), TT,TE,EE+lowE — VERIFY against Planck 2018 VI Table 4
# before quoting; these are the natural same-model literature anchor.
PLANCK_NNU = {'h':(0.6664,0.013), 'omega_b':(0.02224,0.00022), 'omega_cdm':(0.1183,0.0035),
              'n_s':(0.9589,0.0084), 'ln10As':(3.040,0.017), 'tau_reion':(0.0543,0.0073),
              'Neff':(2.92,0.19)}

def loadf(poi):
    f = os.path.join(RES, f'{PREFIX}_{poi}{TAG}.npz')
    return np.load(f, allow_pickle=True) if os.path.exists(f) else None

def getk(d, k, default):
    return np.array(d[k]) if (d is not None and k in d.files) else default

data = {poi: loadf(poi) for poi in ORDER}
present = [p for p in ORDER if data[p] is not None]
pending = [p for p in ORDER if data[p] is None]
print(f'present ({len(present)}/7):', present)
print('pending      :', pending if pending else '(none — all POIs done)')

In [ ]:
# ---- load the SMC posterior (same likelihood+model) for the primary cross-check ----
smc = None
for cand in ['smc_plikfull_neff.npz']:
    p = os.path.join(RES, cand)
    if os.path.exists(p):
        smc = np.load(p, allow_pickle=True); break
SMC = {}
if smc is not None:
    co = [str(x) for x in smc['cosmo_order']]
    mm = np.asarray(smc['marg_mean'], float); ms = np.asarray(smc['marg_std'], float)
    SMC = {k:(m,s) for k,m,s in zip(co, mm, ms)}
    print(f"SMC posterior loaded: logZ={float(smc['logZ']):.2f}, N={smc['N']}, "
          f"wall={float(smc['wall_seconds'])/3600:.1f}h")
    for k in ORDER:
        if k in SMC: print(f'  {k:10s} {SMC[k][0]: .5f} +/- {SMC[k][1]:.5f}')
else:
    print('SMC posterior npz not found — cross-check vs SMC will be skipped.')

In [ ]:
# ---- summary table: best-fit +/- 1sigma, 2sigma, convergence, vs SMC, vs Planck ----
def interval(d):
    """Return (lo1,mid,hi1,lo2,hi2). Prefer stored sigma1/sigma2; else interpolate the
    Delta-chi2=1,4 crossings from the (grid,chi2) curve."""
    s1 = getk(d,'sigma1',None); s2 = getk(d,'sigma2',None)
    grid = np.asarray(d['poi_grid'],float); chi2 = np.asarray(d['chi2'],float)
    if s1 is not None and np.size(s1)==3 and np.all(np.isfinite(s1)):
        lo1,mid,hi1 = [float(x) for x in s1]
    else:
        mid = float(grid[np.argmin(chi2)]); lo1=hi1=np.nan
    lo2,hi2 = (float(s2[0]),float(s2[1])) if (s2 is not None and np.size(s2)==2) else (np.nan,np.nan)
    return lo1,mid,hi1,lo2,hi2

hdr = f"{'POI':10s} {'best':>9s} {'-1s':>8s} {'+1s':>8s} {'conv':>8s} {'max|g|':>9s} {'vsSMC':>7s} {'vsP18':>7s}"
print(hdr); print('-'*len(hdr))
summary = {}
for poi in ORDER:
    d = data[poi]
    if d is None:
        print(f'{poi:10s} {"(pending)":>9s}'); continue
    lo1,mid,hi1,lo2,hi2 = interval(d)
    grid = np.asarray(d['poi_grid'],float); chi2 = np.asarray(d['chi2'],float)
    gn   = getk(d,'gnorm', np.full(len(grid),np.nan)).astype(float)
    conv = getk(d,'converged', np.zeros(len(grid),bool))
    has_cert = (data[poi] is not None and 'converged' in data[poi].files)
    nconv = int(np.nansum(conv)); maxg = float(np.nanmax(gn)) if np.isfinite(gn).any() else np.nan
    pref = PLANCK_NNU.get(poi); sref = SMC.get(poi)
    dsm = (mid-sref[0])/sref[1] if sref else np.nan
    dpl = (mid-pref[0])/pref[1] if pref else np.nan
    cstr = f'{nconv:2d}/{len(grid):<2d}' if has_cert else f'n/a({len(grid)})'
    print(f'{poi:10s} {mid:9.5f} {mid-lo1:8.5f} {hi1-mid:8.5f} {cstr:>8s} {maxg:9.2e} '
          f'{dsm:+6.2f}s {dpl:+6.2f}s')
    summary[poi] = dict(lo1=lo1,mid=mid,hi1=hi1,lo2=lo2,hi2=hi2,nconv=nconv,
                        ntot=len(grid),maxg=maxg,dsm=dsm,dpl=dpl)
print('\nlegend: best = Delta-chi2 minimum; +/-1s = Delta-chi2=1 crossings;')
print('        conv = rows with ||g||_AD < GTOL; vsSMC/vsP18 = (best - ref_mean)/ref_sigma.')
print('        vsP18 ref = Planck 2018 base_nnu (EDIT/VERIFY PLANCK_NNU); vsSMC ref = this-code SMC.')

In [ ]:
# ---- overlay Delta-chi2 profiles (7 panels) ----
fig, axes = plt.subplots(3, 3, figsize=(14, 11)); axes = axes.ravel()
for ax, poi in zip(axes, ORDER):
    d = data[poi]
    if d is None:
        ax.set_title(f'{LABEL[poi]}  (pending)'); ax.axis('off'); continue
    grid = np.asarray(d['poi_grid'],float); chi2 = np.asarray(d['chi2'],float)
    dchi2 = chi2 - np.nanmin(chi2)
    order = np.argsort(grid); grid, dchi2 = grid[order], dchi2[order]
    ax.plot(grid, dchi2, 'o-', ms=4, lw=1.2, color='C0')
    # smooth crossing guide via fine interpolation of the curve
    gg = np.linspace(grid.min(), grid.max(), 600)
    yy = np.interp(gg, grid, dchi2)
    for lev, c in [(1.0,'C3'), (4.0,'C2')]:
        ax.axhline(lev, color=c, ls=':', lw=1, alpha=0.7)
    s = summary.get(poi, {})
    for x in [s.get('lo1'), s.get('hi1')]:
        if x is not None and np.isfinite(x): ax.axvline(x, color='C3', ls='--', lw=1, alpha=0.8)
    if s.get('mid') is not None: ax.axvline(s['mid'], color='k', ls='-', lw=1, alpha=0.6)
    # SMC marginal (Gaussian) for visual comparison
    if poi in SMC:
        m, sd = SMC[poi]
        ax.axvspan(m-sd, m+sd, color='C1', alpha=0.15, label='SMC 1$\\sigma$')
        ax.axvline(m, color='C1', ls='-', lw=1, alpha=0.8)
    ax.set_title(LABEL[poi]); ax.set_xlabel(LABEL[poi]); ax.set_ylabel(r'$\Delta\chi^2$')
    ax.set_ylim(-0.3, 10); ax.grid(alpha=0.25)
    if poi in SMC: ax.legend(fontsize=8, loc='upper center')
for ax in axes[len(ORDER):]: ax.axis('off')
fig.suptitle(f'LCDM+N_eff full-plik profile likelihood (tag {TAG}) — black=min, red dash=1$\\sigma$, '
             f'orange=SMC', y=1.0)
fig.tight_layout(); plt.show()

In [ ]:
# ---- headline: N_eff zoom + profile-vs-SMC-vs-Planck comparison ----
if data['Neff'] is not None:
    d = data['Neff']; grid = np.asarray(d['poi_grid'],float); chi2 = np.asarray(d['chi2'],float)
    dchi2 = chi2 - np.nanmin(chi2); o = np.argsort(grid); grid,dchi2 = grid[o],dchi2[o]
    s = summary['Neff']
    fig, ax = plt.subplots(figsize=(8,5.5))
    ax.plot(grid, dchi2, 'o-', color='C0', label='profile (this work)')
    ax.axhline(1, color='C3', ls=':'); ax.axhline(4, color='C2', ls=':')
    for x,lab in [(s['lo1'],None),(s['hi1'],None)]:
        if np.isfinite(x): ax.axvline(x, color='C3', ls='--', lw=1)
    ax.axvline(s['mid'], color='k', lw=1)
    if 'Neff' in SMC:
        m,sd = SMC['Neff']; ax.axvspan(m-sd,m+sd,color='C1',alpha=0.2,label=f'SMC {m:.3f}±{sd:.3f}')
    if 'Neff' in PLANCK_NNU:
        m,sd = PLANCK_NNU['Neff']; ax.errorbar([m],[8.5],xerr=[[sd],[sd]],fmt='s',color='C4',
                                               capsize=4,label=f'Planck18 base_nnu {m:.2f}±{sd:.2f}')
    ax.axvline(3.044, color='gray', ls='-.', lw=1, alpha=0.7, label='SM $N_{eff}=3.044$')
    ax.set_xlabel(r'$N_{eff}$'); ax.set_ylabel(r'$\Delta\chi^2$'); ax.set_ylim(-0.3,10)
    ax.grid(alpha=0.25); ax.legend(fontsize=9)
    lo,mid,hi = s['lo1'],s['mid'],s['hi1']
    ax.set_title(f'$N_{{eff}}$ profile:  {mid:.3f}  (+{hi-mid:.3f} / -{mid-lo:.3f}) at 1$\\sigma$')
    plt.show()
    print(f'N_eff profile      : {mid:.4f}  +{hi-mid:.4f} / -{mid-lo:.4f}  (1sigma)')
    print(f'  2sigma interval  : [{s["lo2"]:.4f}, {s["hi2"]:.4f}]')
    if 'Neff' in SMC: print(f'N_eff SMC marginal : {SMC["Neff"][0]:.4f} +/- {SMC["Neff"][1]:.4f}')
    print(f'  SM value 3.044 lies at Delta-chi2 = '
          f'{float(np.interp(3.044, grid, dchi2)):.2f}  '
          f'(= {np.sqrt(max(float(np.interp(3.044,grid,dchi2)),0)):.2f} sigma from the min)')
else:
    print('Neff POI not yet written.')

In [ ]:
# ---- convergence / quality gate per POI ----
# FULL-PLIK NOTE (validated on the interactive node 2026-06-29): the AD ||g|| certificate
# FLOORS well above GTOL (~a few, not ~0) because the SPG inner nuisance profile leaves a
# roughness floor in the envelope-theorem gradient. That is EXPECTED for full plik and does
# NOT mean the interval is wrong. The validated convergence criteria are (a) a FINITE,
# in-grid Delta-chi2=1 interval and (b) the sigma1-stability early-stop. So ||g|| is reported
# as INFORMATIONAL and the gate is the finite interval.
for poi in ORDER:
    d = data[poi]
    if d is None:
        continue
    grid = np.asarray(d['poi_grid'], float)
    gn   = getk(d, 'gnorm', np.full(len(grid), np.nan)).astype(float)
    gtol = float(d['gtol']) if 'gtol' in d.files else np.nan
    s = summary.get(poi, {})
    lo1, hi1 = s.get('lo1', np.nan), s.get('hi1', np.nan)
    finite = np.isfinite(lo1) and np.isfinite(hi1)
    onedge = finite and (min(lo1, hi1) <= grid.min() + 1e-9 or max(lo1, hi1) >= grid.max() - 1e-9)
    flag = 'OK  ' if (finite and not onedge) else ('EDGE' if finite else '!!  ')
    ivl  = f'[{lo1:.5f}, {hi1:.5f}]' if finite else 'NOT FINITE'
    print(f'{flag} {poi:10s} 1sig_interval={ivl:24s} '
          f'max|g|_AD={np.nanmax(gn):.2e} (GTOL={gtol:.0e}; floors high for full-plik = expected)')
print('\nGate = FINITE Delta-chi2=1 interval INSIDE the grid (OK).')
print('  EDGE = interval touches a grid edge -> recenter/extend that POI.')
print('  !!   = interval not bracketed (under-converged or off-grid).')
print('  ||g||_AD is INFORMATIONAL for full plik (SPG roughness floor, ~a few >> GTOL); the')
print('  interval rests on the chi2 profile + the sigma1-stability early-stop, not on ||g||<GTOL.')

## How to read this

- **Best-fit ± 1σ** (summary table, red dashed lines) = the Δχ²=1 crossings of the profile;
  **2σ** = Δχ²=4. These are the frequentist intervals the tool delivers.
- **vsSMC** (orange) is the primary cross-check: the SMC posterior is the *same* likelihood,
  model, and ABCMB code, so profile-min and SMC-mean agreeing to a fraction of a σ validates
  the tool. For the **N_eff** direction expect the profile min to sit *below* the SMC marginal
  mean (the N_eff–H0 degeneracy is skewed, so the Bayesian marginal mean is pulled up relative
  to the profile minimum — a real and expected profile-vs-marginal difference, not a bug).
- **vsP18** (`PLANCK_NNU`) is the literature anchor — **edit/verify** those numbers against
  Planck 2018 VI Table 4 (base_nnu, TT,TE,EE+lowE) before quoting.
- **Convergence cell:** the gate is a **finite, in-grid Δχ²=1 interval** (`OK`); `EDGE` = the
  interval touched a grid edge (recenter/extend that POI), `!!` = not bracketed. The AD `||g||`
  is **informational only for full plik** — it floors at ~a few (≫ GTOL) because the SPG inner
  nuisance profile leaves a roughness floor in the envelope-theorem gradient (verified on the
  interactive node 2026-06-29, where the interval was finite and matched the SMC to ~0.7σ while
  `||g||`≈6.9). Convergence rests on the χ² profile + the σ1-stability early-stop, **not** on
  `||g|| < GTOL`. (For comparison, plik-*lite* floors much lower, ~0.1.)
- The job (`55235706`) is resumable: if a POI shows `(pending)`, re-run the load cell once its
  `.npz` appears (`scan/results/profile_prod_ad_<poi>_pfrc.npz`).